# ClusterBP v2: другие модели последовательностей вместо GRU

Почему в последовательной ветке стоит GRU? Сравнение с тремя альтернативами на тех же входах (20 последних измерений, 5 каналов) и в той же схеме обучения:

- **TimeAwareGRU** — GRU с затуханием скрытого состояния по времени между измерениями (идея GRU-D, Che et al. 2018, и DATA-GRU, Tan et al. 2020: учёт нерегулярности ряда);
- **TCN** — свёрточная сеть с причинными расширенными свёртками;
- **Transformer** — небольшой энкодер с вниманием по всем измерениям окна.

Каждая модель сравнивается с GRU и отдельно, и в стекинге с XGB_full (как итоговая модель ClusterBP v2): важно, меняет ли более сложная последовательная модель качество всего ансамбля.
Фолды, внутренняя валидация, прореживание отрицательных и ранняя остановка — как в `clusterbp_v2.ipynb`. GRU и XGB_full берутся из уже обученных моделей фолдов.

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from clusterbp_lib import (RESULTS_DIR, SEED, N_FOLDS, FEATURE_SETS, SEQ_MODELS, load_cached, fold_masks,
                           add_phenotype_columns, load_gru, load_xgb, train_seq_model, gru_logits, sigmoid, logit,
                           best_threshold, boot_eval, ci)

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count())
SEQ_TARGETS = ['A', 'B']
NEW = ['TimeAwareGRU', 'TCN', 'Transformer']

win, X_seq, seq_len, pheno_win, oof = load_cached()
pid = win['pid'].values

In [2]:
store = {t: {} for t in SEQ_TARGETS}
info = []


def put(t, name, te, s, p):
    st = store[t].setdefault(name, [np.full(len(win), np.nan), np.zeros(len(win), np.int8)])
    st[0][te] = s[te]
    st[1][te] = p[te]


t_all = time.time()
for k in range(N_FOLDS):
    tr, va, te = fold_masks(win, k)
    add_phenotype_columns(win, pheno_win, k)
    for t in SEQ_TARGETS:
        y = win[f'y_{t}'].values
        px = load_xgb('full', t, k).predict_proba(win[FEATURE_SETS['full']].values)[:, 1]
        gm, mu, sd, corr = load_gru(t, k)
        seq_probs = {'GRU': sigmoid(gru_logits(gm, X_seq, seq_len, mu, sd) + corr)}
        for name in NEW:
            t0 = time.time()
            m, mu_, sd_, corr_, log = train_seq_model(SEQ_MODELS[name], X_seq[tr], seq_len[tr], y[tr],
                                                      X_seq[va], seq_len[va], y[va], SEED + k)
            seq_probs[name] = sigmoid(gru_logits(m, X_seq, seq_len, mu_, sd_) + corr_)
            info.append({'target': t, 'fold': k, 'model': name, 'эпох': len(log),
                         'параметров': sum(p.numel() for p in m.parameters()), 'секунд': time.time() - t0})
        for name, ps in seq_probs.items():
            thr, _ = best_threshold(y[va], ps[va])
            put(t, name, te, ps, (ps > thr).astype(np.int8))
            st = LogisticRegression(max_iter=3000).fit(np.column_stack([logit(ps[va]), logit(px[va])]), y[va])
            s = st.predict_proba(np.column_stack([logit(ps), logit(px)]))[:, 1]
            thr, _ = best_threshold(y[va], s[va])
            put(t, f'{name} + XGB_full (стекинг)', te, s, (s > thr).astype(np.int8))
    print(f'Фолд {k}: {(time.time() - t_all) / 60:.1f} мин')
pd.DataFrame(info).to_csv(os.path.join(RESULTS_DIR, 'sequence_models_training.csv'), index=False)

Фолд 0: 15.7 мин


Фолд 1: 41.0 мин


Фолд 2: 67.8 мин


Фолд 3: 96.0 мин


Фолд 4: 1449.0 мин


## Результаты

Разности — относительно GRU (для отдельных моделей) и относительно стекинга GRU + XGB_full (для ансамблей); 95% ДИ — бутстреп по пациентам.

In [3]:
rows = []
for t in SEQ_TARGETS:
    y = win[f'y_{t}'].values.astype(float)
    labels = list(store[t])
    models = [(store[t][n][0], store[t][n][1].astype(float)) for n in labels]
    B = boot_eval(y, pid, models, n_boot=1000)
    for j, lab in enumerate(labels):
        ref = 'GRU + XGB_full (стекинг)' if 'стекинг' in lab else 'GRU'
        jr = labels.index(ref)
        d = B[:, j, 1] - B[:, jr, 1]
        rows.append({'target': t, 'модель': lab, 'AUROC': roc_auc_score(y, models[j][0]),
                     'AUPRC': average_precision_score(y, models[j][0]), 'AUPRC 95% ДИ': ci(B[:, j, 1]),
                     'macro_F1': f1_score(y, models[j][1], average='macro'),
                     'ΔAUPRC к ' + ref: '' if lab == ref else f'{np.mean(d):+.4f} {ci(d)}'})
seq = pd.DataFrame(rows)
seq.to_csv(os.path.join(RESULTS_DIR, 'sequence_models.csv'), index=False)
pd.set_option('display.max_colwidth', 60)
display(seq.round(4))
display(pd.DataFrame(info).groupby(['target', 'model'])[['эпох', 'параметров', 'секунд']].mean().round(1))

,target,модель,AUROC,AUPRC,AUPRC 95% ДИ,macro_F1,ΔAUPRC к GRU,ΔAUPRC к GRU + XGB_full (стекинг)
0,A,GRU,0.8724,0.4030,[0.374; 0.430],0.7010,,NaN
1,A,GRU + XGB_full (стекинг),0.8972,0.4514,[0.422; 0.478],0.7156,NaN,
2,A,TimeAwareGRU,0.8762,0.4066,[0.377; 0.433],0.7026,+0.0036 [-0.001; 0.008],NaN
3,A,TimeAwareGRU + XGB_full (стекинг),0.8975,0.4520,[0.423; 0.478],0.7149,NaN,+0.0006 [-0.000; 0.001]
4,A,TCN,0.8749,0.4012,[0.372; 0.428],0.7035,-0.0017 [-0.007; 0.004],NaN
5,A,TCN + XGB_full (стекинг),0.8975,0.4522,[0.423; 0.479],0.7158,NaN,+0.0009 [-0.000; 0.002]
6,A,Transformer,0.8758,0.3985,[0.370; 0.424],0.7010,-0.0045 [-0.013; 0.003],NaN
7,A,Transformer + XGB_full (стекинг),0.8975,0.4518,[0.423; 0.478],0.7160,NaN,+0.0005 [-0.000; 0.001]
8,B,GRU,0.8945,0.6431,[0.626; 0.660],0.7734,,NaN
9,B,GRU + XGB_full (стекинг),0.9107,0.6778,[0.661; 0.694],0.7849,NaN,


эпох  параметров   секунд
target model                                  
A      TCN           12.0     49921.0  16161.2
       TimeAwareGRU  24.0     18050.0    153.5
       Transformer    9.2     68673.0    132.6
B      TCN           10.6     49921.0    300.3
       TimeAwareGRU  18.0     18050.0    235.2
       Transformer    9.2     68673.0    394.7